# Text Summarization using Transformers library

In [20]:
!pip install evaluate
!pip install rouge_score
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [2]:
import transformers
import torch 
from transformers import AutoTokenizer 
from transformers import AutoModelForSeq2SeqLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import get_scheduler
import nltk
from tqdm.auto import tqdm 
import numpy as np 

Importing and loading Google's **MT5-small model** for Text Summarization task

In [3]:
model_checkpoint = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
print(f"Model Name : {model.name_or_path}\nparameters : {model.num_parameters()}")

Model Name : google/mt5-small
parameters : 556291456


In [5]:
raw_dataset = load_dataset("abisee/cnn_dailymail","1.0.0")

**Dataset Overview**

In [6]:
print("-"*100)
print("Dataset Overview")
print(f"Article : {raw_dataset['train']['article'][0]}")
print(f"Highlights : {raw_dataset['train']['highlights'][0]}")
print("-"*100)
print(f"{raw_dataset}")

----------------------------------------------------------------------------------------------------
Dataset Overview
Article : LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or s

**Preprocessing the dataset**

In [7]:
#renameing and removing useless columns
raw_dataset = raw_dataset.rename_column('article','text')
raw_dataset = raw_dataset.rename_column('highlights','summary')
raw_dataset = raw_dataset.remove_columns(['id'])
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 11490
    })
})

In [8]:
raw_dataset['train'] = raw_dataset['train'].select(range(10000))
raw_dataset['test'] = raw_dataset['test'].select(range(2000))
raw_dataset['validation'] = raw_dataset['validation'].select(range(2000))

In [9]:
Max_length = 512 
Max_target_length = 30

def tokenization_function(dataset):
    model_input = tokenizer(dataset['text'],max_length=Max_length,truncation=True)
    labels = tokenizer(dataset['summary'],max_length=Max_target_length,truncation=True)
    
    model_input['labels'] = labels['input_ids']
    return model_input

In [10]:
tokenized_dataset = raw_dataset.map(tokenization_function,batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['text','summary'])
tokenized_dataset

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [11]:
import evaluate 
rouge_score = evaluate.load('rouge')

In [17]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model,padding=True)

In [18]:
tokenized_dataset.set_format('torch')
Batch_size = 8
Train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size = Batch_size,
    shuffle=True,
    collate_fn = data_collator
)
Eval_dataloader = DataLoader(
    tokenized_dataset['validation'],
    batch_size = Batch_size,
    shuffle = False,
    collate_fn = data_collator
)

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
Epochs = 5 
num_steps_per_epochs = len(Train_dataloader)
num_train_steps = Epochs * num_steps_per_epochs
optimizer = torch.optim.AdamW(model.parameters(),lr=0.00002)
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps=0,
    num_training_steps=num_train_steps
)

In [15]:
def postprocess_text(preds, labels):

    preds = preds.cpu().numpy()
    labels = labels.cpu().numpy()

    preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels

In [ ]:
print(f"Training started on {device} for {Epochs} epochs.")
progress_bar = tqdm(range(num_train_steps))

for epoch in range(Epochs):
    model.train()
    for batch in Train_dataloader:
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss 
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    

    model.eval()
    for batch in Eval_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch['input_ids'],
                attention_mask = batch['attention_mask']
            )
        labels = batch['labels']
        decoded_preds, decoded_labels = postprocess_text(generated_tokens, labels)

        rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    result = rouge_score.compute()
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    result = {k: round(v, 4) for k, v in result.items()}
    print(f"[{epoch+1}/{Epochs}] | Rouge Score : {result}")   
